In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.interpolate import interp1d
from scipy.signal import find_peaks, welch, get_window
import os
from scipy.optimize import curve_fit
import time


def load_files(integer):
    #folder_path = "simulation_results_fixed_design"
    #folder_path = os.path.join("..", "pybamm_march2024", "sep22_2024_sim_data_fft_testing_4xp8_extended_part1")
    folder_path = os.path.join("..", "pybamm_march2024", folder_name)
    file_extension = f"_{integer}.npy"
    date_time_str = ""

    for filename in os.listdir(folder_path):
        if filename.endswith(file_extension):
            # Extract the date and time part from the filename
            parts = filename.split('_')
            date_time_str = f"{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}"

            if filename.startswith("params"):
                theta = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("time_data"):
                t = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("voltage_data"):
                v = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("current_data"):
                current = np.load(os.path.join(folder_path, filename))

    return theta, t, v, current, date_time_str
extended_factor = int(input("Please enter the value for extended_factor (should be 4 by default): "))
int_range = int(input("Enter the integer range for the for loop: "))
plot_or_not = int(input("Should you plot or not? (1 for yes, 0 for no): "))
folder_name = input("Please enter the input folder name with the simulations (e.g., sep22_2024_sim_data_fft_testing_4xp8_extended_part1): ")
output_folder_name = input("Please enter the folder name for saving summary statistics (e.g., summary_sep24_2024_sim_data_fft_testing_4xp8_extended_part1): ")
# Create the output folder if it doesn't exist
output_folder_path = os.path.join("..", "pybamm_march2024", output_folder_name)
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

for integer in range(int_range):
    # Load files based on the provided integer
    theta, t, v, current, date_time_str = load_files(integer)
    p1, p2, p3, p4, p5, p6, p7, p8 = theta[9:17]
    
    p8 = p8*extended_factor 
    start_time=time.time()

    # Interpolate the entire dataset

    # Find the size of the largest block

    # Define block durations and repeat to create the full durations
    block_durations_sizing = [p8 + 200, p2 + p3, p5 + p6, p5 + 300]
    full_durations_sizing = np.tile(block_durations_sizing, 5)  # Adjust the repetition count as needed

    # Calculate start and end times for each block
    start_times_sizing = np.cumsum(np.concatenate(([0], full_durations_sizing[:-1])))
    end_times_sizing = start_times_sizing + full_durations_sizing

    # Find indices for start and end times using vectorized operations
    start_indices_sizing = np.searchsorted(t, start_times_sizing, side='right')
    end_indices_sizing = np.searchsorted(t, end_times_sizing, side='right') - 1  # -1 because end index is inclusive

    # Explicitly set the start of the first block to 0
    start_indices_sizing[0] = 0

    # Ensure the end of the last block does not exceed the length of the array
    end_indices_sizing[-1] = min(end_indices_sizing[-1], len(t) - 1)

    # Create the blocks based on the indices found
    tBlocks_sizing = [t[start:end+1] for start, end in zip(start_indices_sizing, end_indices_sizing)]
    vBlocks_sizing = [v[start:end+1] for start, end in zip(start_indices_sizing, end_indices_sizing)]
    iBlocks_sizing = [current[start:end+1] for start, end in zip(start_indices_sizing, end_indices_sizing)]

    # Find the largest block size
    largest_block_size = max(len(block) for block in tBlocks_sizing)
    num_points_chirp = max(len(block) for block in tBlocks_sizing)
    num_points = num_points_chirp * 100
    # Print the largest block size
    #print(f"Size of the largest block: {largest_block_size}")
    #num_points_chirp = largest_block_size
    #print("Num point chirp",num_points_chirp)

    #num_points = num_points_chirp*20000 #where is this coming from? 
    #num_points = len(t)
    t_interp = np.linspace(t[0], t[-1], num_points)
    #t_interp = np.arange(t[0], t[-1] + min_time_step, min_time_step)
    #num_points = int((t[-1] - t[0]) / min_time_step) + 1

    #t_interp = np.linspace(t[0], t[-1], num_points)
    #print(f"Number of interpolation points: {num_points}")
    #print(len(t_interp))
    v_interp_func = interp1d(t, v, kind='linear')
    i_interp_func = interp1d(t, current, kind='linear')

    v_interp = v_interp_func(t_interp)
    i_interp = i_interp_func(t_interp)

    # Break the interpolated data into blocks
    block_durations = [p8, 200, p2, p3, p5, p6, p5, 300]
    full_durations = block_durations * 5  # Adjust the repetition count as needed

    start_times = [0]
    for duration in full_durations[:-1]:  # Exclude the last duration to avoid going out of bounds
        next_start_time = start_times[-1] + duration
        start_times.append(next_start_time)
    end_times = [start + duration for start, duration in zip(start_times, full_durations)]

    def find_index(time_array, time):
        return np.searchsorted(time_array, time, side='right')

    start_indices = [find_index(t_interp, time) for time in start_times]
    end_indices = [find_index(t_interp, time) - 1 for time in end_times]  # -1 because end index is inclusive


    start_indices[0] = 0

    tBlocks = [t_interp[start:end+1] for start, end in zip(start_indices, end_indices)]  # +1 because end index is inclusive
    vBlocks = [v_interp[start:end+1] for start, end in zip(start_indices, end_indices)]
    iBlocks = [i_interp[start:end+1] for start, end in zip(start_indices, end_indices)]

    interp_tBlocks = tBlocks
    interp_vBlocks = vBlocks
    interp_iBlocks = iBlocks
    
    
    #this is the plotting code
    # Load your data as before, and once you have the blocks:
    #colors = plt.cm.viridis(np.linspace(0, 1, len(interp_tBlocks)))  # Generate a color map with distinct colors for each block
    color_options = plt.cm.tab20(np.linspace(0, 1, len(interp_tBlocks)))  # 'tab20' has 20 distinct colors
    color_options = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan',
                 'magenta', 'yellow', 'black', '#FF6347', '#4682B4', '#8B4513', '#32CD32', '#FFD700', '#4B0082', '#00CED1']

    if plot_or_not:
        plt.figure(figsize=(10, 6))  # Create a figure with a specific size

        # Plot each block with a different color
        for i, (t_block, i_block) in enumerate(zip(interp_tBlocks, interp_vBlocks)):
            plt.plot(t_block, i_block, color=color_options[i % len(color_options)], label=f'Block {i + 1}')  # Plot with different colors

        plt.xlabel('Time (t)')
        plt.ylabel('Current (A)')
        plt.title('Current vs. Time for All Blocks')
        #plt.legend()  # Show legend to distinguish blocks
        plt.grid(True)  # Optional: Add grid for better readability
        plt.show()

    frequency_ranges = [(1e-3, 1e-2), (1e-2, 1e-1), (1e-1, 1), (1, 10)]


    def summarize_peak_time_differences(time_uniform, output_voltage_uniform):
        positive_peaks, _ = find_peaks(output_voltage_uniform)
        negative_peaks, _ = find_peaks(-output_voltage_uniform)
        all_peaks = np.sort(np.concatenate((positive_peaks, negative_peaks)))
        time_differences = []
        paired_peaks = []

        for i in range(len(all_peaks) - 1):
            if (all_peaks[i] in positive_peaks and all_peaks[i+1] in negative_peaks) or \
               (all_peaks[i] in negative_peaks and all_peaks[i+1] in positive_peaks):
                time_diff = time_uniform[all_peaks[i+1]] - time_uniform[all_peaks[i]]
                time_differences.append(time_diff)
                paired_peaks.append((all_peaks[i], all_peaks[i+1]))

        #max_time_diff = np.max(time_differences) if time_differences else np.nan
        #min_time_diff = np.min(time_differences) if time_differences else np.nan
        mean_time_diff = np.mean(time_differences) if time_differences else np.nan
        #std_time_diff = np.std(time_differences) if time_differences else np.nan

        return mean_time_diff

    def summarize_impedance_phase(frequency_ranges, impedance, phase_shift, frequencies):
        def average_and_std_over_frequency_range(frequencies, data, frequency_ranges):
            averages = []
            stds = []
            for f_min, f_max in frequency_ranges:
                mask = (frequencies >= f_min) & (frequencies < f_max)
                avg = np.mean(data[mask])
                std = np.std(data[mask])
                averages.append(avg)
                stds.append(std)
            return averages, stds

        average_impedance, std_impedance = average_and_std_over_frequency_range(frequencies, impedance, frequency_ranges)
        average_phase, std_phase = average_and_std_over_frequency_range(frequencies, phase_shift, frequency_ranges)

        return average_impedance, average_phase

    def calculate_power_and_bandwidth(output_voltage_uniform, time_uniform, percentage=0.95):
        sampling_rate = 1 / (time_uniform[1] - time_uniform[0])
        frequencies, psd = welch(output_voltage_uniform, fs=sampling_rate)
        total_power = np.trapz(psd, frequencies)

#         def power_bandwidth(frequencies, psd, percentage=0.95):
#             cumulative_power = np.cumsum(psd) / np.sum(psd)
#             lower_index = np.where(cumulative_power >= (1 - percentage) / 2)[0][0]
#             upper_index = np.where(cumulative_power <= 1 - (1 - percentage) / 2)[0][-1]
#             bandwidth = frequencies[upper_index] - frequencies[lower_index]
#             return frequencies[lower_index], frequencies[upper_index], bandwidth

#         lower_freq, upper_freq, bandwidth = power_bandwidth(frequencies, psd, percentage=percentage)

        return total_power

    def calculate_impedance_and_phase(time, input_current, output_voltage):
        
        
        window = get_window('hann',len(input_current))
        input_current_windowed = input_current*window
        output_voltage_windowed = output_voltage*window
        
        
        n = len(time)
        d = time[1] - time[0]
        input_fft = fft(-input_current)
        voltage_fft = fft(output_voltage)
        frequencies = fftfreq(n, d)
        
        #CHECK - NANs here or in FFT? Check input currents and voltage 
        #PLOT INPUT CURRENT AND VOLTAGE SIGNALS AND PLOT THE FFTS and check for NaNs in individual signals
        # if it's a single poitn giving NaNs - drop it and interpolate 
        impedance = np.abs(voltage_fft / input_fft)
        phase_shift = np.angle(voltage_fft) - np.angle(input_fft)
        
            # Check if there are frequencies between 1e-3 and 1e-2
#         frequency_range = (1e-3, 1e-2)
#         frequencies_in_range = frequencies[(frequencies >= frequency_range[0]) & (frequencies < frequency_range[1])]

#         if frequencies_in_range.size > 0:
#             print(f"Frequencies between {frequency_range[0]} and {frequency_range[1]} Hz are present.")
#             print(f"Number of frequencies in range: {frequencies_in_range.size}")
#         else:
#             print(f"No frequencies found between {frequency_range[0]} and {frequency_range[1]} Hz.")

        return frequencies, impedance, phase_shift

    def combined_summary_chirp(time_uniform, input_current_uniform, output_voltage_uniform, frequency_ranges, percentage=0.95):
        
#         window = get_window('hann',len(input_current_uniform))
#         input_current_windowed = input_current_uniform*window
        
#         input_fft = fft(input_current_windowed)
#         output_fft = fft(output_voltage_uniform)
#         frequencies = fftfreq(len(input_current_uniform), d= (time_uniform[1]-time_uniform[0]))
        
#         impedance = np.abs(output_fft/input_fft)
#         phase_shift = np.angle(output_fft) - np.angle(input_fft)
        
        #mean_time_diff = summarize_peak_time_differences(time_uniform,output_voltage_uniform)
        
        #average_impedance, average_phase = summarize_impedance_phase(frequency_ranges,impedance,phase_shift, frequencies)

        
        mean_time_diff = summarize_peak_time_differences(time_uniform, output_voltage_uniform)
        frequencies, impedance, phase_shift = calculate_impedance_and_phase(time_uniform, input_current_uniform, output_voltage_uniform)
        average_impedance, average_phase = summarize_impedance_phase(frequency_ranges, impedance, phase_shift, frequencies)
        total_power = calculate_power_and_bandwidth(output_voltage_uniform, time_uniform, percentage)

        results = {
            "peak_time_differences": {
                "mean_time_diff": mean_time_diff
            },
            "impedance_phase_summary": {
                "average_impedance": average_impedance,
                "average_phase": average_phase
            },
            "power_bandwidth_summary": {
                "total_power": total_power
            }
        }

        return results

    # for block % 8 == 1
    def exponential_decay(t, k):
        return np.exp(-k * t)
    def negative_exponential_decay(t, k):
        return -np.exp(-k * t)
    def process_special_block(v_block, t_block):
        # First voltage value (V0) is just the first value in v_block
        # Perform the curve fitting
        # def fit_exponential_decay(t, V):
        #     V0 = V[0]
        #     t_values = t - t[0]  # Time differences relative to the first time point
        #     v_values = V / V0  # Normalize by V0
        #     popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #     k = popt[0]
        #     return V0, k
        # V0, k = fit_exponential_decay(t_block,v_block)
        V0 = v_block[0]

        # Fit the difference to an exponential decay V = V0 * exp(-k * t) to find k
        #t_values = t_block[:2] - t_block[0]  # Time differences relative to the first time point
        #v_values = v_block[:2] / V0  # Normalize by V0
        #popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #k = popt[1]
        #print("popt length", len(popt))
        #print("popt:", popt)
        #print("vblock [1]", v_block[1])
        #print("vblock[0]", v_block[0])
        k = ( (v_block[1] - v_block[0]) / (t_block[1] - t_block[0]) )
        k = k / V0#because when you linearize the slope has a V0 term to it 
        #print("k", k)

        # Find the final voltage value in the block
        final_V = v_block[-1]

        return V0, k, final_V

    def process_exponential_decay_block(v_block, t_block):
        #num_points = int(np.ceil(len(v_block) / 4))
        #print("This is for the block 3 calculation")
        #def fit_exponential_decay(t, V, is_positive_decay):
            #V0 = V[0]
            #t_values = t - t[0]  # Time differences relative to the first time point
            #v_values = V / V0  # Normalize by V0
            #t_values = t - t[0]
            #v_values = V
            #popt, _ = curve_fit(exponential_decay, t_values, v_values)
            #initial_guess = [1e-5]
            #bounds = (0, 1)
            #if is_positive_decay:
            #    popt, pcov = curve_fit(exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)
            #else:
            #    popt, pcov = curve_fit(negative_exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)

            #popt, pcov = curve_fit(exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)
            #k = popt[0]
            #return k, pcov

        #print("This is the t_values:", )
        V0 = v_block[0]
        #t_values = t_block[:num_points]  # Time differences relative to the first time point
        #print("This is the t_values:", t_values)
        #v_values = v_block[:num_points]
        final_V = v_block[-1]
        is_positive_decay = final_V < V0 
        #print("This is the v_values:", v_values)
        #k, pcov = fit_exponential_decay(t_values,v_values, is_positive_decay)
        #popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #k = popt[0]

        return V0, k, final_V

    def calculate_summary_statistics(t_block, v_block, i_block):
        # Calculate resistance
        resistance = np.abs( (v_block[0] - v_block[-1]) / np.mean(i_block))

        # Calculate dV/dt
        dV_dt = np.gradient(v_block, t_block)
        #dV_dt = np.mean(dV_dt)
        # Calculate charge (Q) by integrating current over time
        Q = np.cumsum(-i_block) * (t_block[1] - t_block[0])

        # Calculate dV/dQ
        dV_dQ = np.gradient(v_block, Q)
        # Example: replace NaNs with zero
        dV_dQ = np.nan_to_num(dV_dQ, nan=1e5)
        dV_dQ = np.where(dV_dQ > 1e10, 1e5, dV_dQ)
        # if(dV_dQ>1e10):
        #     dV_dQ = 1e5

        #dV_dQ = np.mean(dV_dQ)

        return resistance, np.mean(dV_dt), np.mean(dV_dQ), Q
    # Example usage

    #print("v Blocks", interp_vBlocks[7])
    #print("t Blocks", interp_tBlocks[7])
    #V0, k, final_V, pcov = process_exponential_decay_block(interp_vBlocks[7], interp_tBlocks[7])
    #print(f"V0: {V0}, k: {k}, final_V: {final_V}, pcov: {pcov}")
    


    #define time_uniform, input_current_uniform, output_voltage_uniform


    results_blocks = []
    results_array = []


    for blockNumber, (t_block, v_block, i_block) in enumerate(zip(interp_tBlocks, interp_vBlocks, interp_iBlocks)):
        if blockNumber % 8 == 0:
            #apply a hann window to reduce spectral leakage
            window = get_window('hann', len(i_block))
            i_block_windowed = i_block*window
            current_fft = fft(i_block_windowed)
            freqs = fftfreq(len(i_block_windowed), d=(t_block[1]-t_block[0]))
            magnitude = np.abs(current_fft)
            frequency_results = {}
            for lower,upper in frequency_ranges:
                indices_in_range = (np.abs(freqs) >= lower) & (np.abs(freqs)  < upper)
                range_freqs = freqs[indices_in_range]
                range_magnitudes = magnitude[indices_in_range]
                
                frequency_results[f"{lower}-{upper} Hz"] = {
                    "frequencies": range_freqs, 
                    "magnitudes": range_magnitudes
                }
            results = combined_summary_chirp(t_block,i_block,v_block,frequency_ranges)
            results["frequency_analysis"] = frequency_results
            results_blocks.append((blockNumber,results))
            results_array.append(results)
            # results = combined_summary_chirp(t_block, i_block, v_block, frequency_ranges)
            # results_blocks.append((blockNumber, results))
            # results_array.append(results)
        elif blockNumber % 8 == 1:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 2:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 3:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 4:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 5:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 6:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 8 == 7:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)

    results_array = []


    nu_array = []
    
    #this is the one that actually gets saved
    for blockNumber, results in results_blocks:
        if blockNumber % 8 == 0:
            
            print(f"This is block: {blockNumber}")

            print(f"Block Number: {blockNumber}")
            print("Peak Time Differences Summary:")
            for key, value in results["peak_time_differences"].items():
                #print(f"  {key.replace('_', ' ').title()}: {value:.3f} s")
                nu_array.append(value)

            #print("Impedance and Phase Summary:")
            for (f_min, f_max), avg_imp, avg_phase in zip(frequency_ranges, results["impedance_phase_summary"]["average_impedance"], results["impedance_phase_summary"]["average_phase"]):
                #print(f"  Frequency range {f_min} to {f_max} Hz:")
                #print(f"    Average Impedance: {avg_imp:.3f} Ohms")
                nu_array.append(avg_imp)
                #print(f"    Std Impedance: {std_imp:.3f} Ohms")
                
                #print(f"    Average Phase: {avg_phase:.3f} Radians")
                nu_array.append(avg_phase)
             #   print(f"    Std Phase: {std_phas:.3f} Radians")
                #print(f"This is block: {blockNumber}")

           # print("Power and Bandwidth Summary:")
            for key, value in results["power_bandwidth_summary"].items():
            #    print(f"  {key.replace('_', ' ').title()}: {value:.3f} Hz")
                nu_array.append(value)


            values = [
                results["peak_time_differences"]["mean_time_diff"],
                *results["impedance_phase_summary"]["average_impedance"],
                *results["impedance_phase_summary"]["average_phase"],
                results["power_bandwidth_summary"]["total_power"]
            ]
            #results_array.append(results["peak_time_differences"]["max_time_diff"])

        elif blockNumber % 8 == 1:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  V0: {results['V0']:.3f}")
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            #print(f"  k: {results['k']:.9f}")
            #print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 2:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  Resistance: {results['resistance']:.3f} Ohms")
            #print(f"  dV/dt: {results['dV_dt']}")
            #print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 3:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  V0: {results['V0']:.3f}")
            #print(f"  k: {results['k']:.9f}")
            #print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 4:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  Resistance: {results['resistance']:.3f} Ohms")
            #print(f"  dV/dt: {results['dV_dt']}")
            #print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 5:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  V0: {results['V0']:.3f}")
            #print(f"  k: {results['k']:.9f}")
            #print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 6:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  Resistance: {results['resistance']:.3f} Ohms")
            #print(f"  dV/dt: {results['dV_dt']}")
            #print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 8 == 7:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  V0: {results['V0']:.3f}")
            #print(f"  k: {results['k']:.15f}")
            #print(f"  Final V: {results['final_V']:.3f}")
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            values = [results['V0'], results['k'], results['final_V']]
            #print(f"This is block: {blockNumber}")

    # Convert results_array to a numpy array and print it
    #results_array = np.array(results_array)
    # Initialize a counter for the total number of summary statistics
    total_summary_statistics = 0

    # Iterate through results_blocks and count the summary statistics
    for blockNumber, results in results_blocks:
        if blockNumber % 8 == 0:
            # Count the number of summary statistics for combined_summary
            total_summary_statistics += len(results["peak_time_differences"]) \
                                        + len(results["impedance_phase_summary"]["average_impedance"]) \
                                        + len(results["impedance_phase_summary"]["average_phase"]) \
                                        + len(results["power_bandwidth_summary"])
        elif blockNumber % 8 in [1, 3, 5, 7]:
            # Count the number of summary statistics for exponential decay
            total_summary_statistics += len(results) 
        elif blockNumber % 8 in [2, 4, 6]:
            # Count the number of summary statistics for resistance, mean dV/dt, and mean dV/dQ
            total_summary_statistics += len(results)

    # Print the total number of summary statistics
    #print(f"Total number of summary statistics stored in results_blocks: {total_summary_statistics}")

    nu_array_np = np.array(nu_array)
    
    #folder_path = "summary_statistics_fixed_design"
    #folder_path = os.path.join("..", "pybamm_march2024", "summary_sep24_2024_sim_data_fft_testing_4xp8_extended_part1")

    #if not os.path.exists(folder_path):
    #    os.makedirs(folder_path)

    # Define the new file name based on the extracted date, time, and integer value
    file_name = f"summary_stats_{date_time_str}_{integer}.npy"
    file_path = os.path.join(output_folder_path, file_name)

    # Save the numpy array to the file
    np.save(file_path, nu_array_np)

    #print(f"Array saved to {file_path}")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time for integer {integer}: {elapsed_time} seconds")


Please enter the value for extended_factor (should be 4 by default):  4
Enter the integer range for the for loop:  2034
Should you plot or not? (1 for yes, 0 for no):  0
Please enter the input folder name with the simulations (e.g., sep22_2024_sim_data_fft_testing_4xp8_extended_part1):  sep22_2024_sim_data_fft_testing_4xp8_extended_part1
Please enter the folder name for saving summary statistics (e.g., summary_sep24_2024_sim_data_fft_testing_4xp8_extended_part1):  summary_sep22_2024_sim_data_fft_testing_4xp8_extended_part1


This is block: 0
Block Number: 0
Peak Time Differences Summary:
This is block: 8
Block Number: 8
Peak Time Differences Summary:
This is block: 16
Block Number: 16
Peak Time Differences Summary:
This is block: 24
Block Number: 24
Peak Time Differences Summary:
This is block: 32
Block Number: 32
Peak Time Differences Summary:
Elapsed time for integer 0: 0.2474365234375 seconds
This is block: 0
Block Number: 0
Peak Time Differences Summary:
This is block: 8
Block Number: 8
Peak Time Differences Summary:
This is block: 16
Block Number: 16
Peak Time Differences Summary:
This is block: 24
Block Number: 24
Peak Time Differences Summary:
This is block: 32
Block Number: 32
Peak Time Differences Summary:
Elapsed time for integer 1: 0.08337545394897461 seconds
This is block: 0
Block Number: 0
Peak Time Differences Summary:
This is block: 8
Block Number: 8
Peak Time Differences Summary:
This is block: 16
Block Number: 16
Peak Time Differences Summary:
This is block: 24
Block Number: 24
Peak Time D

In [ ]:
# Define the file path
file_path = "../pybamm_march2024_copy/summary_statistics_fixed_design_new_and_cleaned_July_2024/summary_stats_data_20240710_185827_0.npy"

# Load the .npy file
data = np.load(file_path)

# Check the length and shape of the data
length = len(data)
shape = data.shape

print(f"Length: {length}")
print(f"Shape: {shape}")

In [ ]:
import os
import numpy as np

# Define the relative path to the target directory
target_directory = "../pybamm_march2024_copy/summary_statistics_fixed_design_new_and_cleaned_July_2024"

# Get the list of all .npy files in the target directory
npy_files = [f for f in os.listdir(target_directory) if f.endswith('.npy')]

# Initialize an empty list to store the data
data_list = []

# Loop through each .npy file and load the data
for npy_file in npy_files:
    file_path = os.path.join(target_directory, npy_file)
    data = np.load(file_path)
    data_list.append(data)

# Convert the list of arrays into a single numpy array
combined_data = np.vstack(data_list)

# Check the shape of the combined data
combined_shape = combined_data.shape

print(f"Shape of the combined data: {combined_shape}")


In [ ]:
# Find which columns have NaNs
columns_with_nans = np.any(np.isnan(combined_data), axis=0)
columns_with_nans_indices = np.where(columns_with_nans)[0]

# Check which rows in the columns with NaNs contain NaNs
rows_with_nans_info = {}
for col in columns_with_nans_indices:
    rows_with_nans = np.isnan(combined_data[:, col])
    rows_with_nans_info[col] = np.where(rows_with_nans)[0]

# Print the information about rows with NaNs for each column with NaNs
for col, rows in rows_with_nans_info.items():
    print(f"Column {col} has NaNs in rows: {rows}")

In [ ]:
summary_stat_descriptions = [
    "Max Time Diff Block 0", "Min Time Diff Block 0", "Mean Time Diff Block 0", "Std Time Diff Block 0",
    "Avg Impedance (0.001 to 0.01 Hz) Block 0", "Std Impedance (0.001 to 0.01 Hz) Block 0", "Avg Phase (0.001 to 0.01 Hz) Block 0", "Std Phase (0.001 to 0.01 Hz) Block 0",
    "Avg Impedance (0.01 to 0.1 Hz) Block 0", "Std Impedance (0.01 to 0.1 Hz) Block 0", "Avg Phase (0.01 to 0.1 Hz) Block 0", "Std Phase (0.01 to 0.1 Hz) Block 0",
    "Avg Impedance (0.1 to 1 Hz) Block 0", "Std Impedance (0.1 to 1 Hz) Block 0", "Avg Phase (0.1 to 1 Hz) Block 0", "Std Phase (0.1 to 1 Hz) Block 0",
    "Avg Impedance (1 to 10 Hz) Block 0", "Std Impedance (1 to 10 Hz) Block 0", "Avg Phase (1 to 10 Hz) Block 0", "Std Phase (1 to 10 Hz) Block 0",
    "Total Power Block 0", "Lower Freq Block 0", "Upper Freq Block 0", "Bandwidth Block 0",
    "V0 Block 1", "k Block 1", "Final V Block 1", "Resistance Block 2", "dV/dt Block 2", "dV/dQ Block 2",
    "V0 Block 3", "k Block 3", "Final V Block 3", "Resistance Block 4", "dV/dt Block 4", "dV/dQ Block 4",
    "V0 Resistance Block 5", "k Resistance Block 5", "Final V Resistance Block 5", "Resistance Resistance Block 6", "dV/dt Resistance Block 6", "dV/dQ Resistance Block 6",
    "V0 Resistance Block 7", "k Resistance Block 7", "Final V Resistance Block 7"
]

# indices to be removed (modulo 45)
bad_modulo_indices = [0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 22, 23]


new_summary_stat_descriptions = [desc for idx, desc in enumerate(summary_stat_descriptions) if idx not in bad_modulo_indices]



for i, desc in enumerate(new_summary_stat_descriptions):
    print(f"Column index {i} corresponds to variable '{desc}'")

In [ ]:
# Print information about columns with NaNs and their corresponding variable descriptions
for col in columns_with_nans_indices:
    variable_desc = new_summary_stat_descriptions[col % len(new_summary_stat_descriptions)]
    rows_with_nans = rows_with_nans_info[col]
    print(f"Column index {col} ({variable_desc}) has NaNs in rows: {rows_with_nans}")

In [ ]:
# Check for NaNs in the data
nan_exists = np.isnan(data).any()

if nan_exists:
    print("The data contains NaNs.")
else:
    print("The data does not contain any NaNs.")

In [ ]:
# Check for NaNs in the data and get their indices
nan_indices = np.where(np.isnan(data))

print("Indices of NaNs:", nan_indices)

# Apply the modulo operation to each array in the tuple of indices
nan_indices_mod_31 = tuple(idx % 31 for idx in nan_indices)

print("Indices of NaNs modulo 31:", nan_indices_mod_31)
#the NaNs are Column 'Avg Impedance (0.001 to 0.01 Hz) Block 0' and  'Avg Phase (0.001 to 0.01 Hz) Block 0' for cycles 3,4,5 (counting from 1)

In [ ]:
summary_stat_descriptions = [
    "Max Time Diff Block 0", "Min Time Diff Block 0", "Mean Time Diff Block 0", "Std Time Diff Block 0",
    "Avg Impedance (0.001 to 0.01 Hz) Block 0", "Std Impedance (0.001 to 0.01 Hz) Block 0", "Avg Phase (0.001 to 0.01 Hz) Block 0", "Std Phase (0.001 to 0.01 Hz) Block 0",
    "Avg Impedance (0.01 to 0.1 Hz) Block 0", "Std Impedance (0.01 to 0.1 Hz) Block 0", "Avg Phase (0.01 to 0.1 Hz) Block 0", "Std Phase (0.01 to 0.1 Hz) Block 0",
    "Avg Impedance (0.1 to 1 Hz) Block 0", "Std Impedance (0.1 to 1 Hz) Block 0", "Avg Phase (0.1 to 1 Hz) Block 0", "Std Phase (0.1 to 1 Hz) Block 0",
    "Avg Impedance (1 to 10 Hz) Block 0", "Std Impedance (1 to 10 Hz) Block 0", "Avg Phase (1 to 10 Hz) Block 0", "Std Phase (1 to 10 Hz) Block 0",
    "Total Power Block 0", "Lower Freq Block 0", "Upper Freq Block 0", "Bandwidth Block 0",
    "V0 Block 1", "k Block 1", "Final V Block 1", "Resistance Block 2", "dV/dt Block 2", "dV/dQ Block 2",
    "V0 Block 3", "k Block 3", "Final V Block 3", "Resistance Block 4", "dV/dt Block 4", "dV/dQ Block 4",
    "V0 Resistance Block 5", "k Resistance Block 5", "Final V Resistance Block 5", "Resistance Resistance Block 6", "dV/dt Resistance Block 6", "dV/dQ Resistance Block 6",
    "V0 Resistance Block 7", "k Resistance Block 7", "Final V Resistance Block 7"
]

# indices to be removed (modulo 45)
bad_modulo_indices = [0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 22, 23]
num_columns = 225

# modulo 45
indices_to_remove = [i for i in range(num_columns) if i % 45 in bad_modulo_indices]

# create a new list of summary stat descriptions after removing bad indices
new_summary_stat_descriptions = [desc for idx, desc in enumerate(summary_stat_descriptions) if idx not in bad_modulo_indices]


for i, desc in enumerate(new_summary_stat_descriptions):
    print(f"Column index {i} corresponds to variable '{desc}'")